# NyaayKhel — 01: Pose Extraction Test

**Purpose:** Smoke-test the YOLOv8-pose pipeline on a kabaddi video clip.
Confirm that multi-person keypoints extract cleanly before building any model on top.

**This notebook:**
1. Installs YOLOv8 (Ultralytics)
2. Downloads a sample kabaddi clip from YouTube
3. Runs YOLOv8-pose inference on every frame
4. Saves annotated frames and a skeleton-overlay video
5. Benchmarks inference speed and person-detection count
6. Exports raw keypoint sequences as numpy arrays (preview of Phase B format)

**Exit gate:** Keypoints extract cleanly on a kabaddi clip, ≥ 2 persons detected per frame, inference ≥ 5fps on Colab GPU.

## Cell 1: Install Dependencies

In [ ]:
!pip install -q ultralytics yt-dlp opencv-python-headless matplotlib

# Verify
import ultralytics
print(f'Ultralytics version: {ultralytics.__version__}')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Cell 2: Set Up Paths

In [ ]:
import os

# Mount Drive if you want to save outputs persistently
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/NyaayKhel'
else:
    BASE_DIR = '/content/NyaayKhel'

SAMPLE_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'samples')
OUTPUT_DIR = os.path.join(BASE_DIR, 'docs', 'pose_test_outputs')

for d in [SAMPLE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Sample clip dir: {SAMPLE_DIR}')
print(f'Output dir:      {OUTPUT_DIR}')

## Cell 3: Download a Sample Kabaddi Clip

In [ ]:
import yt_dlp
import subprocess

# ── CONFIGURATION ────────────────────────────────────────────────────────────
# We download a SHORT clip from YouTube for the smoke test.
# Priority: side-angle (~90°), good player visibility, multiple players in frame.
# 
# Option A: Let yt-dlp search for a match clip automatically
# Option B: Paste a specific YouTube URL below (recommended if you found a good one)

USE_SPECIFIC_URL = False  # Set True and fill VIDEO_URL to use a specific video
VIDEO_URL = ''            # e.g. 'https://www.youtube.com/watch?v=XXXXXXXXXXX'

SEARCH_QUERY = 'ytsearch1:kabaddi tournament side angle full match 2023'
SAMPLE_PATH_FULL = os.path.join(SAMPLE_DIR, 'sample_full.mp4')
SAMPLE_PATH_CLIP = os.path.join(SAMPLE_DIR, 'sample_clip_30s.mp4')  # 30-sec trimmed clip
# ─────────────────────────────────────────────────────────────────────────────

ydl_opts = {
    'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best[height<=720]',
    'outtmpl': SAMPLE_PATH_FULL,
    'quiet': False,
    'noplaylist': True,
    'match_filter': yt_dlp.utils.match_filter_func('duration < 3600'),
}

target = VIDEO_URL if USE_SPECIFIC_URL else SEARCH_QUERY
print(f'Downloading from: {target}')

if not os.path.exists(SAMPLE_PATH_FULL):
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(target, download=True)
    print(f'Downloaded: {SAMPLE_PATH_FULL}')
else:
    print(f'Already exists: {SAMPLE_PATH_FULL}')

# Trim to 30 seconds starting at 60s (skip intro/dead time)
TRIM_START_SEC = 60
TRIM_DURATION_SEC = 30

print(f'Trimming to {TRIM_DURATION_SEC}s clip starting at {TRIM_START_SEC}s...')
subprocess.run([
    'ffmpeg', '-y',
    '-ss', str(TRIM_START_SEC),
    '-i', SAMPLE_PATH_FULL,
    '-t', str(TRIM_DURATION_SEC),
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '20',
    '-an', '-vf', 'scale=640:-2',
    SAMPLE_PATH_CLIP
], check=True, capture_output=True)

print(f'Sample clip ready: {SAMPLE_PATH_CLIP}')

# Get clip info
probe = subprocess.run(
    ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', SAMPLE_PATH_CLIP],
    capture_output=True, text=True
)
import json
streams = json.loads(probe.stdout).get('streams', [])
for s in streams:
    if s.get('codec_type') == 'video':
        print(f"Clip: {s['width']}x{s['height']}, {s.get('avg_frame_rate','?')} fps, duration: {float(s.get('duration',0)):.1f}s")

## Cell 4: Load YOLOv8-pose Model

In [ ]:
from ultralytics import YOLO

# yolov8n-pose = nano — smallest, fastest. Good for low-end device benchmarking.
# Alternatives if you want better accuracy: yolov8s-pose, yolov8m-pose
# For Android TFLite export, nano is the right choice.
MODEL_NAME = 'yolov8n-pose.pt'

model = YOLO(MODEL_NAME)  # downloads automatically on first run
print(f'Model loaded: {MODEL_NAME}')
print(f'Task: {model.task}')
print(f'Device: {next(model.model.parameters()).device}')

## Cell 5: Run Pose Extraction on the Sample Clip + Benchmark Speed

In [ ]:
import cv2
import numpy as np
import time
from collections import defaultdict

# ── CONFIG ────────────────────────────────────────────────────────────────────
# Process every Nth frame to benchmark different effective fps targets
PROCESS_EVERY_N_FRAMES = 1  # 1 = every frame; 3 = every 3rd frame
CONFIDENCE_THRESHOLD = 0.5  # Min person confidence to keep
KEYPOINT_CONF_THRESHOLD = 0.3  # Min keypoint confidence to keep (else set to 0)
MAX_FRAMES = 300  # Cap frames for quick benchmark (None = process all)
# ─────────────────────────────────────────────────────────────────────────────

cap = cv2.VideoCapture(SAMPLE_PATH_CLIP)
fps_video = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'Video FPS: {fps_video:.1f} | Total frames: {total_frames}')

# Storage for keypoint sequences (to preview Phase B data format)
# Shape per frame: (num_persons, 17_keypoints, 3) — x, y, confidence
keypoint_sequences = []   # list of per-frame arrays
person_counts = []
inference_times = []

frame_idx = 0
processed_count = 0

YOLO_KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if MAX_FRAMES and frame_idx >= MAX_FRAMES:
        break
    frame_idx += 1

    if frame_idx % PROCESS_EVERY_N_FRAMES != 0:
        continue

    t0 = time.perf_counter()
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    t1 = time.perf_counter()

    inference_times.append(t1 - t0)

    result = results[0]
    n_persons = len(result.boxes) if result.boxes is not None else 0
    person_counts.append(n_persons)

    # Extract keypoints for this frame: shape (N_persons, 17, 3)
    if result.keypoints is not None and n_persons > 0:
        kp_data = result.keypoints.data.cpu().numpy()  # (N, 17, 3)
        # Zero out low-confidence keypoints
        kp_data[kp_data[:, :, 2] < KEYPOINT_CONF_THRESHOLD] = 0
        keypoint_sequences.append(kp_data)
    else:
        keypoint_sequences.append(None)

    processed_count += 1
    if processed_count % 30 == 0:
        avg_t = np.mean(inference_times[-30:]) * 1000
        print(f'  Frame {frame_idx} | persons: {n_persons} | avg inference: {avg_t:.1f}ms')

cap.release()

print(f'\n--- RESULTS ---')
print(f'Frames processed: {processed_count}')
print(f'Avg inference time: {np.mean(inference_times)*1000:.1f}ms ({1/np.mean(inference_times):.1f} fps effective)')
print(f'Min inference time: {np.min(inference_times)*1000:.1f}ms')
print(f'Max inference time: {np.max(inference_times)*1000:.1f}ms')
print(f'Avg persons detected per frame: {np.mean(person_counts):.1f}')
print(f'Max persons in a frame: {np.max(person_counts)}')
print(f'Frames with 0 persons: {sum(1 for c in person_counts if c == 0)}')
print(f'Frames with ≥2 persons: {sum(1 for c in person_counts if c >= 2)}')

## Cell 6: Visualise Keypoints — Skeleton Overlay on Sample Frames

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Re-run model on a few frames and display annotated output
DISPLAY_N_FRAMES = 6
FRAME_INDICES_TO_SHOW = [10, 30, 60, 90, 120, 150][:DISPLAY_N_FRAMES]

cap = cv2.VideoCapture(SAMPLE_PATH_CLIP)
annotated_frames = []

frame_idx = 0
target_set = set(FRAME_INDICES_TO_SHOW)

while cap.isOpened() and len(annotated_frames) < DISPLAY_N_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx in target_set:
        results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
        # YOLOv8 has built-in annotated plot
        annotated = results[0].plot()
        annotated_frames.append((frame_idx, cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)))
    frame_idx += 1

cap.release()

# Display grid
n = len(annotated_frames)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = np.array(axes).flatten()

for i, (fidx, img) in enumerate(annotated_frames):
    axes[i].imshow(img)
    axes[i].set_title(f'Frame {fidx}')
    axes[i].axis('off')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle('YOLOv8-pose Keypoints on Kabaddi Footage\n(Smoke test — NyaayKhel Phase A)', fontsize=14)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'pose_test_frames.png')
plt.savefig(save_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Cell 7: Export Annotated Skeleton Video

In [ ]:
# Export a short annotated video with skeleton overlays.
# Useful for visual review and for the architecture diagram / deck.

VIDEO_OUT_PATH = os.path.join(OUTPUT_DIR, 'pose_annotated.mp4')
MAX_EXPORT_FRAMES = 150  # ~5 seconds at 30fps

cap = cv2.VideoCapture(SAMPLE_PATH_CLIP)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_out = min(fps_video, 30.0)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(VIDEO_OUT_PATH, fourcc, fps_out, (w, h))

frame_idx = 0
print(f'Exporting annotated video to: {VIDEO_OUT_PATH}')

while cap.isOpened() and frame_idx < MAX_EXPORT_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)
    annotated = results[0].plot()
    out.write(annotated)
    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f'  Exported frame {frame_idx}/{MAX_EXPORT_FRAMES}')

cap.release()
out.release()
print(f'\nAnnotated video saved: {VIDEO_OUT_PATH}')
print(f'Duration: ~{frame_idx/fps_out:.1f}s at {fps_out:.0f}fps')

## Cell 8: Preview Keypoint Sequence Data Format

This shows the exact data format that Phase B (dataset builder) will consume.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Demonstration of the normalisation + windowing approach used in Phase B.
# This uses the keypoint sequences collected during Cell 5.
# ---------------------------------------------------------------------------

WINDOW_SIZE = 30  # frames per classification window (at 10fps → 3 seconds of action)
MAX_PERSONS = 2   # we track at most 2 players (raider + closest defender)
N_KEYPOINTS = 17  # YOLOv8-pose outputs

def normalize_keypoints(kp_array, frame_w, frame_h):
    """
    Normalise x,y to [0,1] relative to frame dimensions.
    confidence values kept as-is.
    kp_array shape: (N_persons, 17, 3)  —  x, y, conf
    """
    normed = kp_array.copy()
    normed[:, :, 0] /= frame_w  # x
    normed[:, :, 1] /= frame_h  # y
    return normed

def pad_persons(kp_array, max_persons):
    """
    Pad or truncate to exactly max_persons.
    If fewer than max_persons detected, pad with zeros.
    """
    n = kp_array.shape[0]
    if n >= max_persons:
        return kp_array[:max_persons]
    pad = np.zeros((max_persons - n, N_KEYPOINTS, 3))
    return np.concatenate([kp_array, pad], axis=0)

# Build one example window from the collected sequences
cap_temp = cv2.VideoCapture(SAMPLE_PATH_CLIP)
w_frame = int(cap_temp.get(cv2.CAP_PROP_FRAME_WIDTH))
h_frame = int(cap_temp.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_temp.release()

print(f'Frame dimensions: {w_frame}x{h_frame}')
print(f'Window size: {WINDOW_SIZE} frames | Max persons: {MAX_PERSONS} | Keypoints: {N_KEYPOINTS}')

# Build a sample window from collected keypoints
window_frames = []
for kps in keypoint_sequences[:WINDOW_SIZE]:
    if kps is None or len(kps) == 0:
        frame_vec = np.zeros((MAX_PERSONS, N_KEYPOINTS, 3))
    else:
        normed = normalize_keypoints(kps, w_frame, h_frame)
        padded = pad_persons(normed, MAX_PERSONS)
        frame_vec = padded
    window_frames.append(frame_vec)

while len(window_frames) < WINDOW_SIZE:
    window_frames.append(np.zeros((MAX_PERSONS, N_KEYPOINTS, 3)))

window_array = np.stack(window_frames)  # shape: (30, 2, 17, 3)

# Flatten to (30, 2*17*3) = (30, 102) for GRU input
flat_window = window_array.reshape(WINDOW_SIZE, -1)  # (30, 102)

print(f'\nWindow shape (raw):      {window_array.shape}  →  (frames, persons, keypoints, [x,y,conf])')
print(f'Window shape (flattened): {flat_window.shape}  →  (frames, features) for GRU input')
print(f'Feature vector size per frame: {flat_window.shape[1]}')
print(f'  = {MAX_PERSONS} persons × {N_KEYPOINTS} keypoints × 3 (x, y, conf)')

# Save sample window for inspection
np.save(os.path.join(OUTPUT_DIR, 'sample_window.npy'), flat_window)
print(f'\nSample window saved to: {OUTPUT_DIR}/sample_window.npy')
print('→ This is the input format for the GRU/TCN classifier in Phase B.')

## Cell 9: Export YOLOv8-pose to TFLite (for Android)

This is done once here to verify the export path works. The model will be bundled in the Android app.

In [ ]:
import os

TFLITE_EXPORT_DIR = os.path.join(BASE_DIR, 'model')
os.makedirs(TFLITE_EXPORT_DIR, exist_ok=True)

print('Exporting YOLOv8n-pose to TFLite...')
print('This may take 1–3 minutes.')

# Export to TFLite int8 (quantized) — better for low-end Android devices
# imgsz must match expected input; 320 is a good balance of speed vs accuracy
exported_path = model.export(
    format='tflite',
    imgsz=320,      # smaller = faster on device; 320 good for kabaddi footage at 640px wide
    int8=False,     # Set True for int8 quantization (needs calibration data — do in Phase C)
    half=False,     # float32 for now; half (fp16) only available on some GPU delegates
    simplify=True,
    dynamic=False,  # Static batch size for TFLite
)

print(f'\nTFLite model exported to: {exported_path}')

# Copy to model dir
import shutil
dest = os.path.join(TFLITE_EXPORT_DIR, 'yolov8n_pose.tflite')
if str(exported_path) != dest:
    shutil.copy(str(exported_path), dest)
    print(f'Copied to: {dest}')

size_mb = os.path.getsize(dest) / (1024 * 1024)
print(f'Model size: {size_mb:.1f} MB')
print()
print('NOTE: For Android integration (Phase C), copy this .tflite file to:')
print('  android/app/src/main/assets/yolov8n_pose.tflite')

## Cell 10: Speed Benchmark at Different Frame Rates

Simulate the 5fps / 10fps modes planned for low-end device mitigation.

In [ ]:
import time
import cv2
import numpy as np

# Load a batch of frames for benchmark
N_BENCHMARK_FRAMES = 50
frames_for_bench = []
cap = cv2.VideoCapture(SAMPLE_PATH_CLIP)
while len(frames_for_bench) < N_BENCHMARK_FRAMES:
    ret, f = cap.read()
    if not ret:
        break
    frames_for_bench.append(f)
cap.release()

print(f'Benchmarking on {len(frames_for_bench)} frames...\n')

results_table = []
for stride in [1, 2, 3, 6]:  # process every Nth frame = 30/N effective fps
    frames_to_process = frames_for_bench[::stride]
    times = []
    for frame in frames_to_process:
        t0 = time.perf_counter()
        _ = model(frame, conf=0.5, verbose=False)
        times.append(time.perf_counter() - t0)

    avg_ms = np.mean(times) * 1000
    effective_fps = 1 / np.mean(times)
    video_fps_equiv = fps_video / stride  # equivalent video fps processed

    results_table.append({
        'stride': stride,
        'video_fps_equiv': video_fps_equiv,
        'avg_inference_ms': avg_ms,
        'effective_fps': effective_fps,
    })

print(f'{'Stride':>8} | {'Video fps equiv':>16} | {'Avg inference (ms)':>20} | {'Inference fps':>14}')
print('-' * 70)
for r in results_table:
    print(f"{r['stride']:>8} | {r['video_fps_equiv']:>16.1f} | {r['avg_inference_ms']:>20.1f} | {r['effective_fps']:>14.1f}")

print()
print('NOTE: This benchmark runs on COLAB GPU — on-device (Android, low-end CPU/GPU) will be 5–15x slower.')
print('Target on-device: ≥5 fps inference. Plan to use stride=3+ on low-end devices.')
print('The app includes a configurable PROCESS_EVERY_N_FRAMES setting for exactly this.')

## Cell 11: Exit Gate Summary

In [ ]:
import os

checks = [
    ('Sample clip downloaded',         os.path.exists(SAMPLE_PATH_CLIP)),
    ('Annotated frame grid saved',     os.path.exists(os.path.join(OUTPUT_DIR, 'pose_test_frames.png'))),
    ('Annotated video saved',          os.path.exists(os.path.join(OUTPUT_DIR, 'pose_annotated.mp4'))),
    ('TFLite model exported',          os.path.exists(os.path.join(BASE_DIR, 'model', 'yolov8n_pose.tflite'))),
    ('Sample keypoint window saved',   os.path.exists(os.path.join(OUTPUT_DIR, 'sample_window.npy'))),
    ('≥2 persons detected per frame',  np.mean([c >= 2 for c in person_counts]) >= 0.5),
    ('Inference ≥5fps on Colab GPU',   (1/np.mean(inference_times)) >= 5),
]

print('=' * 50)
print('PHASE A EXIT GATE — Pose Extraction Test')
print('=' * 50)

all_pass = True
for label, passed in checks:
    status = '✓ PASS' if passed else '✗ FAIL'
    if not passed:
        all_pass = False
    print(f'  {status}  {label}')

print()
if all_pass:
    print('✅ ALL CHECKS PASSED — Phase A complete. Proceed to Phase B.')
    print()
    print('Next steps:')
    print('  1. Commit this notebook (outputs cleared) to GitHub')
    print('  2. Run 00_data_collection.ipynb to download 150+ training clips')
    print('  3. Upload clips to CVAT/Label Studio for labeling')
    print('  4. Then run 02_dataset_builder.ipynb')
else:
    print('❌ SOME CHECKS FAILED — debug before proceeding.')
    print('Common fixes:')
    print('  - If <2 persons per frame: try a different clip with clearer side-angle view')
    print('  - If TFLite export fails: check Ultralytics version, try without int8=True')
    print('  - If inference <5fps: already on Colab GPU? Check torch.cuda.is_available()')